# OmniVoice 快速入门

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

> 英文笔记本：[OmniVoice.ipynb](../OmniVoice.ipynb)

本笔记本演示 [OmniVoice](https://github.com/k2-fsa/OmniVoice) 的基本用法——支持 600+ 语言的大规模多语言零样本 TTS。

**目录：**
1. 安装
2. 方案 A — Gradio 演示（交互 Web UI，无需写代码）
3. 方案 B — Python API
   - 3.1 加载模型
   - 3.2 语音克隆
   - 3.3 声音设计
   - 3.4 自动选声


## 1. 安装

Colab 已提供兼容的 PyTorch + CUDA 环境，只需安装 OmniVoice。


In [ ]:
!pip install omnivoice

## 2. 方案 A — Gradio 演示

启动带公网链接的交互 Web UI。`--share` 会创建临时公网 URL，可在任意浏览器访问。

> **若希望直接使用 Python API，请跳到下方方案 B。**


In [ ]:
!omnivoice-demo --share

## 3. 方案 B — Python API

### 3.1 加载模型


In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

### 3.2 语音克隆

用一段较短（约 3–10 秒）的参考音频克隆音色。可上传自己的 `ref.wav` 或任意音频文件。

`ref_text` 为可选项——若省略，模型会用 Whisper ASR 自动转写参考音频。


In [ ]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 声音设计

用说话人属性描述目标音色，无需参考音频。

支持属性：性别、年龄、音高、风格（耳语）、英语口音、中文方言。完整列表见 [docs/zh/voice-design.md](voice-design.md)。


In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 自动选声

由模型自动选择音色——无需参考音频或 instruct。


In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))